#### Importing required libraries 

In [38]:
from utils import Hetero_Data_Processor_Filter_on_Test_since_first_post # Required class for GNN Incremental training
import pandas as pd
from torch import nn
from torch_geometric.nn import HANConv
import torch.nn.functional as F
import numpy as np
import mlflow
import torch
import torch.nn as nn
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
warnings.filterwarnings("ignore")

#### Testing a single load 

In [5]:
event_name ="charlie_hebdo"

In [6]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=1000)
data = processor.process()
data['id'].y = data['id'].y.long()

In [7]:
data

HeteroData(
  id={
    x=[1473, 106],
    y=[1473],
    train_mask=[1473],
    val_mask=[1473],
    test_mask=[1473],
  },
  reply_user_id={ x=[13463, 104] },
  (id, retweet, reply_user_id)={ edge_index=[2, 13463] },
  (reply_user_id, rev_retweet, id)={ edge_index=[2, 13463] }
)

In [8]:
class HAN(nn.Module):

    """
    Heterogeneous Graph Attention Network (HAN) model with two HANConv layers.

    This model is designed for heterogeneous graphs where nodes and edges may
    have different types. It uses hierarchical attention mechanisms to learn
    node representations by aggregating semantic information from multiple
    relations. After two stages of relational attention, the model outputs
    predictions for the `'id'` node type.

    Parameters
    ----------
    dim_in : int
        Input feature dimension shared across node types.
    dim_out : int
        Output feature dimension, typically the number of prediction classes.
    dim_h : int, optional (default=64)
        Hidden dimension used in each HANConv layer.
    heads : int, optional (default=4)
        Number of attention heads in each HANConv layer.

    Attributes
    ----------
    han : HANConv
        First hierarchical attention convolution layer.
    han2 : HANConv
        Second hierarchical attention convolution layer for deeper semantic aggregation.
    linear : nn.Linear
        Final linear projection applied to `'id'` node embeddings.

    Forward Inputs
    --------------
    x_dict : dict[str, torch.Tensor]
        Dictionary mapping node types to feature matrices.
    edge_index_dict : dict[str, torch.Tensor]
        Dictionary mapping edge types to adjacency information.

    Returns
    -------
    torch.Tensor
        Output predictions/logits for `'id'` nodes with shape
        [num_id_nodes, dim_out].
    """
    
    def __init__(self, dim_in, dim_out, dim_h=64, heads=4):
        super().__init__()
        self.han = HANConv(dim_in, dim_h, heads=heads,dropout=0.2, metadata=data.metadata())
        self.han2 = HANConv(dim_h, dim_h, heads=heads, dropout=0.2, metadata=data.metadata())
        self.linear = nn.Linear(dim_h, dim_out)

    def forward(self, x_dict, edge_index_dict):
        out = self.han(x_dict, edge_index_dict)
        out = self.han2(out, edge_index_dict)
        out = self.linear(out['id'])
        return out
    

In [9]:
def evaluate(model, data, mask_names):

    """
    Evaluate a graph classification model using masked subsets of the data.

    This function runs the model in evaluation mode, computes predictions,
    filters them using one or multiple masks from the input dataset, and
    returns common binary classification metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model producing class logits from graph inputs.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data structure containing:
        - `x_dict`: dictionary of node feature matrices
        - `edge_index_dict`: dictionary of edge connectivity
        - `'id'` node type with attributes `y` and boolean masks
          (e.g., 'train_mask', 'val_mask', 'test_mask')
    mask_names : str or list[str]
        Name(s) of mask attributes to evaluate on. If multiple masks
        are provided, they are combined using logical OR.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple containing:
        - acc : float
            Accuracy score.
        - precision : float
            Proportion of predicted positives that are correctly classified.
        - recall : float
            True positive rate.
        - auc : float
            ROC-AUC score based on predicted class probabilities.

    Notes
    -----
    - Metrics are computed only on masked nodes.
    - If ROC-AUC cannot be computed due to a single class present in labels,
      a value of 0.0 is returned.
    """


    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    preds = out.argmax(dim=-1)
    labels = data['id'].y

    if isinstance(mask_names, str):
        mask = data['id'][mask_names]
    else:
        mask = torch.zeros_like(data['id'].y, dtype=torch.bool)
        for name in mask_names:
            mask |= data['id'][name]

    preds_masked = preds[mask]
    labels_masked = labels[mask]
    probs = out[mask][:, 1]  # Fixed: out is a tensor

    acc = accuracy_score(labels_masked.cpu(), preds_masked.cpu())
    precision = precision_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)
    recall = recall_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)

    try:
        auc = roc_auc_score(labels_masked.cpu(), probs.detach().cpu())
    except ValueError:
        auc = 0.0

    return acc, precision, recall, auc

In [10]:
def evaluate(model, data, mask_names):
    """
    Evaluate a graph classification model using masked subsets of the data
    and return both aggregate metrics and a detailed DataFrame.

    Parameters
    ----------
    model : torch.nn.Module
    data : torch_geometric.data.HeteroData
    mask_names : str or list[str]

    Returns
    -------
    tuple(float, float, float, float, pd.DataFrame)
        - acc
        - precision (macro)
        - recall (macro)
        - auc
        - df (node-level predictions)
    """

    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)  # <-- FIX HERE
        preds = out.argmax(dim=1)
        probs = out[:, 1]

    labels = data['id'].y

    # --- Build mask ---
    if isinstance(mask_names, str):
        mask = data['id'][mask_names]
    else:
        mask = torch.zeros_like(labels, dtype=torch.bool)
        for name in mask_names:
            mask |= data['id'][name]

    # --- Apply mask ---
    true = labels[mask]
    pred = preds[mask]
    prob = probs[mask]

    # --- Metrics ---
    acc = accuracy_score(true.cpu(), pred.cpu())
    precision = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)

    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except ValueError:
        auc = 0.0

    # --- DataFrame ---
    df = pd.DataFrame({
        "node_id": mask.nonzero(as_tuple=False).view(-1).cpu().numpy(),
        "true_label": true.cpu().numpy(),
        "prediction": pred.cpu().numpy(),
        "prob": prob.cpu().numpy()
    })

    return acc, precision, recall, auc, df

In [11]:







def train(model, data, optimizer, epochs=100):

    """
    Train a graph neural network on masked node labels and monitor performance.

    This function performs a full training loop where the loss is computed only
    over nodes marked by the `'train_mask'` attribute in the heterogeneous
    graph's `'id'` node type. At each epoch, training metrics are logged, and
    validation metrics are evaluated periodically to track generalization.

    Parameters
    ----------
    model : torch.nn.Module
        The GNN model to be trained. Must accept heterogeneous node features
        (`x_dict`) and edge connectivity (`edge_index_dict`) in its forward pass.
    data : torch_geometric.data.HeteroData
        A heterogeneous graph containing:
        - `x_dict`: node feature dictionaries
        - `edge_index_dict`: adjacency per edge type
        - `'id'` node labels stored in `.y`
        - masks such as `'train_mask'`, `'val_mask'`, `'test_mask'`
    optimizer : torch.optim.Optimizer
        The optimizer used to update model weights.
    epochs : int, optional (default=100)
        Number of training epochs.

    Notes
    -----
    - Loss is computed using cross-entropy over masked nodes.
    - Metrics include accuracy, precision, recall, and ROC-AUC.
    - Validation performance is printed every 10 epochs.
    - At the end of training, results are evaluated on a combined
      validation + test mask for a final performance estimate.

    Returns
    -------
    None
        This function prints training and evaluation logs but does not return a value.
    """

    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x_dict, data.edge_index_dict)
        mask = data['id'].train_mask
        #out_id = out['id']
        loss = F.cross_entropy(out[mask], data['id'].y[mask])
        #loss = cross_entropy(out_id[data['id'].train_mask], data['id'].y[data['id'].train_mask])
        loss.backward()
        optimizer.step()

        # Train metrics
        acc, precision, recall, auc,df_probs_train = evaluate(model, data, 'train_mask')
        print(f"[Epoch {epoch:03d}] Train - Acc: {acc:.4f} | Prec: {precision:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")

        # Val metrics every 10 epochs
        if epoch % 10 == 0:
            acc_val, prec_val, recall_val, auc_val,df_probs_val = evaluate(model, data, 'val_mask')
            print(f"[Epoch {epoch:03d}] Val   - Acc: {acc_val:.4f} | Prec: {prec_val:.4f} | Recall: {recall_val:.4f} | AUC: {auc_val:.4f}")

    print("\nFinal Evaluation (Val + Test):")
    acc_final, prec_final, recall_final, auc_final,df_probs_all = evaluate(model, data, ['val_mask', 'test_mask'])
    print(f"[Final] Val+Test - Acc: {acc_final:.4f} | Prec: {prec_final:.4f} | Recall: {recall_final:.4f} | AUC: {auc_final:.4f}")

    return  acc_final, prec_final, recall_final, auc_final,df_probs_all




#### Example  training

In [13]:

model = HAN(dim_in=-1, dim_out=2)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

In [18]:
acc_final, prec_final, recall_final, auc_final,df_probs_all = train(model, data, optimizer, epochs=100)


[Epoch 001] Train - Acc: 0.9165 | Prec: 0.9171 | Recall: 0.9154 | AUC: 0.9781
[Epoch 002] Train - Acc: 0.9182 | Prec: 0.9186 | Recall: 0.9172 | AUC: 0.9786
[Epoch 003] Train - Acc: 0.9199 | Prec: 0.9204 | Recall: 0.9187 | AUC: 0.9790
[Epoch 004] Train - Acc: 0.9199 | Prec: 0.9207 | Recall: 0.9185 | AUC: 0.9799
[Epoch 005] Train - Acc: 0.9199 | Prec: 0.9211 | Recall: 0.9183 | AUC: 0.9805
[Epoch 006] Train - Acc: 0.9215 | Prec: 0.9223 | Recall: 0.9203 | AUC: 0.9809
[Epoch 007] Train - Acc: 0.9265 | Prec: 0.9264 | Recall: 0.9261 | AUC: 0.9814
[Epoch 008] Train - Acc: 0.9282 | Prec: 0.9282 | Recall: 0.9276 | AUC: 0.9819
[Epoch 009] Train - Acc: 0.9249 | Prec: 0.9253 | Recall: 0.9239 | AUC: 0.9824
[Epoch 010] Train - Acc: 0.9249 | Prec: 0.9253 | Recall: 0.9239 | AUC: 0.9828
[Epoch 010] Val   - Acc: 0.7829 | Prec: 0.7851 | Recall: 0.7839 | AUC: 0.8665
[Epoch 011] Train - Acc: 0.9265 | Prec: 0.9272 | Recall: 0.9254 | AUC: 0.9836
[Epoch 012] Train - Acc: 0.9332 | Prec: 0.9333 | Recall: 0.9326 

#### Setting MLflow Experiment

In [39]:
from datetime import date

today = date.today()
formatted_today = today.strftime("%Y-%m-%d")

In [40]:

mlflow.set_experiment(f"HAN {formatted_today} {event_name}")

2026/07/12 18:16:27 INFO mlflow.tracking.fluent: Experiment with name 'HAN 2026-07-12 charlie_hebdo' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/mlruns/129', creation_time=1783880187653, experiment_id='129', last_update_time=1783880187653, lifecycle_stage='active', name='HAN 2026-07-12 charlie_hebdo', tags={}, workspace='default'>

#### Loading dataset statistics to get the final time cut 

In [41]:
import pandas as pd
df_posts_by_time_cut = pd.read_pickle(f'replies_{event_name}.pkl')

df_posts_by_time_cut['min_since_fst_post'] = round(
            (df_posts_by_time_cut['time'] - df_posts_by_time_cut['time'].min()).dt.total_seconds() / 60, 2)


In [42]:
df_metrics = df_posts_by_time_cut[['id','time','rumour','min_since_fst_post']].drop_duplicates().sort_values(by='time')

In [43]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=1000)
data = processor.process()


In [44]:

start = df_metrics.iloc[int(data['id'].train_mask.sum()):].min_since_fst_post.min()
end = df_metrics.iloc[int(data['id'].train_mask.sum()):].min_since_fst_post.max()
duration = end-start
experiment_time= duration+60



In [12]:
start

np.float64(268.53)

In [23]:
end

np.float64(599.37)

In [24]:
experiment_time

np.float64(390.84000000000003)

In [46]:
def compute_metrics_custom(df, prob_col='prob', target_col='rumour'):
    df = df.copy()
    df = df.sort_values(prob_col, ascending=False).reset_index(drop=True)

    total_frauds = df[target_col].sum()
    total_records = df.shape[0]
    n = len(df)

    # ✅ Bins from 1% to 100% in 1% steps
    bins = np.arange(0.01, 1.01, 0.01)

    results = []

    for p in bins:
        cutoff = int(np.ceil(n * p))
        subset = df.iloc[:cutoff]

        frauds = subset[target_col].sum()
        records = len(subset)

        results.append({
            'percentile': round(p * 100, 0),
            'records': records,
            'frauds_captured': frauds,
            'capture_rate': frauds / total_frauds if total_frauds > 0 else 0,
            'bad_rate': frauds / records if records > 0 else 0,
            'false_positive_rate': (records - frauds) / total_records if records > 0 else 0
        })

    df_out = (
        pd.DataFrame(results)
        .drop_duplicates(subset='percentile')
        .sort_values('percentile')
        .reset_index(drop=True)
    )

    return df_out

In [39]:

previous_node_count = 0  # Start with no nodes

for time_cut in np.linspace(0+15, int(experiment_time), 50):
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    data = processor.process()
    data['id'].y = data['id'].y.long()

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Set up model and training
    model = HAN(dim_in=-1, dim_out=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    
    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 201):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)
            loss = F.cross_entropy(out[data['id'].train_mask], data['id'].y[data['id'].train_mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc,_= evaluate(model, data, 'train_mask')
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Precision: {train_prec:.4f}")
    
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
    
    
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc,df_probs = evaluate(model, data, ['val_mask', 'test_mask'])

        current_df_metrics = df_metrics.iloc[int(data['id'].train_mask.sum()):int(data['id'].train_mask.sum()\
                                                                          +data['id'].val_mask.sum()+data['id'].test_mask.sum())]
        current_df_metrics['prob'] = df_probs['prob'].to_numpy()
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)
        mlflow.log_param('new_post',False)
        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_{event_name}_HAN.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_{event_name}_HAN.csv")

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)


=== Time Cut: 15.0 minutes ===
[Epoch 100] Train Loss: 0.2156 | Train Recall: 0.8094 | Train Precision: 0.8342
[Epoch 200] Train Loss: 0.1075 | Train Recall: 0.9126 | Train Precision: 0.9381
[Final Val+Test] Acc: 0.8000 | Prec: 0.4000 | Recall: 0.5000 | AUC: 1.0000
New Instances: 5
New Precision: 0.4000 | New Recall: 0.5000

=== Time Cut: 45.06122448979592 minutes ===
[Epoch 100] Train Loss: 0.1996 | Train Recall: 0.8422 | Train Precision: 0.8321
[Epoch 200] Train Loss: 0.0947 | Train Recall: 0.9370 | Train Precision: 0.9387
[Final Val+Test] Acc: 1.0000 | Prec: 1.0000 | Recall: 1.0000 | AUC: 1.0000
New Instances: 3
New Precision: 1.0000 | New Recall: 1.0000

=== Time Cut: 75.12244897959184 minutes ===
[Epoch 100] Train Loss: 0.2075 | Train Recall: 0.8322 | Train Precision: 0.8335
[Epoch 200] Train Loss: 0.1077 | Train Recall: 0.9117 | Train Precision: 0.9336
No new instances to evaluate.
[Final Val+Test] Acc: 0.8750 | Prec: 0.4375 | Recall: 0.5000 | AUC: 0.8571
New Instances: 0
New Pr

In [47]:
new_posts_times = np.sort(
    np.unique(
        np.ceil(
            df_metrics[
                (df_metrics.min_since_fst_post >= start) &
                (df_metrics.min_since_fst_post <= end)
            ].min_since_fst_post - start
        )
    )
)

In [48]:
new_posts_times = [col for col in new_posts_times if col > 10]

In [49]:
def evaluate_metrics(model, data, mask):


    """
    Compute evaluation metrics for a model on a masked node subset.

        The function performs prediction using the trained model, extracts
    predictions over a specific boolean mask, and computes standard
    classification metrics including accuracy, macro precision, macro
    recall, and ROC-AUC.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model that outputs logits for the `'id'` node type.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data containing node features, edge connections,
        labels, and a boolean mask to filter evaluation nodes.
    mask : torch.Tensor or list[bool]
        Boolean mask selecting the subset of nodes to evaluate.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple of:
        - acc : float
        
            Accuracy score.
        - prec : float
            Macro-averaged precision.
        - recall : float
            Macro-averaged recall.
        - auc : float
            ROC-AUC score based on probability of the positive class.

    Notes
    -----
    - Evaluation is performed inside a `torch.no_grad()` block to disable gradient tracking.
    - If ROC-AUC computation fails (e.g., only one class present), a value of 0.0 is returned.
    """


    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)
        preds = out.argmax(dim=-1)
        labels = data['id'].y

    true = data['id'].y[mask]
    pred = preds[mask]
    prob = probs[mask]

    acc = accuracy_score(true.cpu(), pred.cpu())
    prec = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except:
        auc = 0.0

    # --- Build DataFrame ---
    df = pd.DataFrame({
        "node_id": mask.nonzero(as_tuple=False).view(-1).cpu().numpy(),
        "true_label": true.cpu().numpy(),
        "prob": prob.cpu().numpy()
    })

    return acc, prec, recall, auc, df

In [ ]:

previous_node_count = 0  # Start with no nodes


for time_cut in new_posts_times:
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    data = processor.process()
    data['id'].y = data['id'].y.long()


    torch.manual_seed(42)
    
    y = data['id'].y
    
    train_mask = data['id'].train_mask
    test_mask = data['id'].test_mask
    
    # Train positive rate
    train_idx = torch.where(train_mask)[0]
    train_pos = (y[train_idx] == 1).sum().item()
    train_neg = (y[train_idx] == 0).sum().item()
    
    train_rate = train_pos / (train_pos + train_neg)
    
    print(f"Train positive rate: {train_rate:.4f}")
    
    # Test positives/negatives
    test_idx = torch.where(test_mask)[0]
    
    test_pos = test_idx[y[test_idx] == 1]
    test_neg = test_idx[y[test_idx] == 0]
    
    print(f"Original test: {len(test_pos)} positives, {len(test_neg)} negatives")
    
    # Number of positives needed
    desired_pos = int(round(train_rate * len(test_neg) / (1 - train_rate)))
    
    desired_pos = min(desired_pos, len(test_pos))
    
    perm = torch.randperm(len(test_pos))
    sampled_pos = test_pos[perm[:desired_pos]]
    
    # New test mask
    new_test_mask = torch.zeros_like(test_mask)
    new_test_mask[test_neg] = True
    new_test_mask[sampled_pos] = True
    
    data['id'].test_mask = new_test_mask
    
    # Verify
    final_idx = torch.where(new_test_mask)[0]
    final_rate = (y[final_idx] == 1).float().mean().item()
    
    print(f"Final test size: {len(final_idx)}")
    print(f"Final positives: {(y[final_idx]==1).sum().item()}")
    print(f"Final negatives: {(y[final_idx]==0).sum().item()}")
    print(f"Final positive rate: {final_rate:.4f}")


    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Set up model and training
    model = HAN(dim_in=-1, dim_out=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    
    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 201):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)
            loss = F.cross_entropy(out[data['id'].train_mask], data['id'].y[data['id'].train_mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc,_= evaluate(model, data, 'train_mask')
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Precision: {train_prec:.4f}")
    
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
            continue
    
    
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc,df_probs = evaluate(model, data, ['val_mask', 'test_mask'])

        current_df_metrics = df_metrics.iloc[int(data['id'].train_mask.sum()):int(data['id'].train_mask.sum()\
                                                                          +data['id'].val_mask.sum()+data['id'].test_mask.sum())]
        current_df_metrics['prob'] = df_probs['prob'].to_numpy()
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)

        
        _, _, _, _,df_probs_new = evaluate_metrics(model, data, torch.tensor(final_mask))
        new_df_metrics = current_df_metrics.iloc[-final_mask.sum():]
        new_df_metrics['prob'] = df_probs_new.to_numpy()
        new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)
        
        #print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
        
        #print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)
        mlflow.log_param('new_post',True)
        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_new_posts_{event_name}_HAN.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_new_posts_{event_name}_HAN.csv")

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)


=== Time Cut: 11.0 minutes ===
Train positive rate: 0.1529
Original test: 0 positives, 2 negatives
Final test size: 2
Final positives: 0
Final negatives: 2
Final positive rate: 0.0000


In [22]:
import numpy as np
import pandas as pd

previous_node_count = 0
results = []

for time_cut in new_posts_times:

    print(f"\n=== Time Cut: {time_cut} ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(
        file_path_replies,
        file_path_posts,
        time_cut=time_cut
    )

    data = processor.process()

    current_node_count = data['id'].x.shape[0]

    new_nodes = np.arange(previous_node_count, current_node_count)

    if len(new_nodes) == 0:
        previous_node_count = current_node_count
        continue

    edge_index = data['id', 'retweet', 'reply_user_id'].edge_index.cpu().numpy()

    src = edge_index[0]   # id nodes
    dst = edge_index[1]   # reply_user_id nodes

    ###############################################################
    # Edges originating from NEW id nodes
    ###############################################################

    edge_mask = np.isin(src, new_nodes)

    new_edges = edge_mask.sum()

    ###############################################################
    # Degree of each new node
    ###############################################################

    degrees = np.bincount(src[edge_mask], minlength=current_node_count)

    avg_degree = degrees[new_nodes].mean()

    ###############################################################
    # Connected new nodes
    ###############################################################

    connected_new_nodes = np.unique(src[edge_mask])

    pct_connected = len(connected_new_nodes) / len(new_nodes)

    ###############################################################

    results.append({
        "time_cut": time_cut,
        "new_nodes": len(new_nodes),
        "edges_from_new_nodes": int(new_edges),
        "avg_degree": float(avg_degree),
        "pct_connected": pct_connected,
    })

    print(f"New nodes: {len(new_nodes)}")
    print(f"Edges: {new_edges}")
    print(f"Average degree: {avg_degree:.2f}")
    print(f"Connected nodes: {pct_connected:.2%}")

    previous_node_count = current_node_count

df = pd.DataFrame(results)

print(df)


=== Time Cut: 11.0 ===
New nodes: 1404
Edges: 13029
Average degree: 9.28
Connected nodes: 100.00%

=== Time Cut: 14.0 ===
New nodes: 1
Edges: 1
Average degree: 1.00
Connected nodes: 100.00%

=== Time Cut: 911.0 ===
New nodes: 3
Edges: 15
Average degree: 5.00
Connected nodes: 100.00%

=== Time Cut: 913.0 ===
New nodes: 1
Edges: 2
Average degree: 2.00
Connected nodes: 100.00%

=== Time Cut: 919.0 ===
New nodes: 1
Edges: 4
Average degree: 4.00
Connected nodes: 100.00%

=== Time Cut: 924.0 ===
New nodes: 1
Edges: 11
Average degree: 11.00
Connected nodes: 100.00%

=== Time Cut: 928.0 ===
New nodes: 1
Edges: 1
Average degree: 1.00
Connected nodes: 100.00%

=== Time Cut: 931.0 ===
New nodes: 2
Edges: 2
Average degree: 1.00
Connected nodes: 100.00%

=== Time Cut: 933.0 ===

=== Time Cut: 937.0 ===

=== Time Cut: 938.0 ===
New nodes: 1
Edges: 1
Average degree: 1.00
Connected nodes: 100.00%

=== Time Cut: 941.0 ===

=== Time Cut: 942.0 ===
New nodes: 1
Edges: 2
Average degree: 2.00
Connected no